# Create LMT Awards (Lietuvos mokslo taryba / Research Council of Lithuania)

Creates Research Council of Lithuania awards from LMT's public project register **Spektras**
(https://spektras.lmt.lt/), via its built-in per-search Excel export (`rodyti.php`). The
tracker's `projektai.lmt.lt` is the login-walled application system (its public-reports page is
empty), so it is not used.

**4,225 projects, 2009-2026** (local run 2026-09-30), ongoing + completed, activity groups:
A national research programmes / commissioned research, B researcher-initiated projects (MIP,
ERC-level, breakthrough ideas, ...), C international/bilateral programmes (national shares).
Excluded: D1 researcher training (postdocs, student research, doctoral support: exported per
sub-programme with *application* numbers and mixed name order), D2 mobility, D3
publication/event support, D4 infrastructure.

Fields: contract number, project leader, title, institution, partner institution/country,
science area/field, period (years only), funds per year and total (EUR). For completed
projects the total is funds *used*; for ongoing projects funds *allocated*. No start/end dates
in the source: `start_year`/`end_year` from the period, dates NULL. Total EUR 567M.

**`funder_award_id` (runbook 2.1.1):** the contract number (`Sutarties Nr.`): `S-MIP-24-105`,
`MIP-047/2014`, ESF `09.3.3-LMT-K-712-...`: the forms citing works use (F4320322689 shells:
1,606 `S-XXX-yy-n`, 911 `XXX-nnn/yyyy`, 729 ESF numbers). 2,438 existing shell rows share an id
with this ingest. Application numbers (`P-MIP-...`) are not in the register.

**Prerequisites:** run `scripts/local/lmt_to_s3.py` (uploads
`s3://openalex-ingest/awards/lmt/lmt_projects.parquet`, 1.4 shrink guard).

**Funder (Path A):** funder_id `4320322689`, Lietuvos Mokslo Taryba, ror 02gs16m83,
doi 10.13039/501100004504 (read from `openalex.funders.funders`).

**Priority:** `486`.

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.lmt_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/lmt/lmt_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects, COUNT(DISTINCT funder_award_id) as distinct_ids,
       COUNT(amount_total) as with_amount
FROM openalex.awards.lmt_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.lmt_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.lmt_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320322689;

## Step 2: Create LMT Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.lmt_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320322689  -- Lietuvos Mokslo Taryba
),

g AS (
    SELECT
        r.*,
        CASE WHEN NULLIF(TRIM(r.lead_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(r.lead_given_name), ''),
            'family_name', NULLIF(TRIM(r.lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(r.institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead
    FROM openalex.awards.lmt_raw r
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(NULLIF(TRIM(g.title), ''), 'Duomenys neteikiami') as display_name,  -- 'data not provided' placeholder
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CASE WHEN TRY_CAST(g.amount_total AS DOUBLE) > 0 THEN TRY_CAST(g.amount_total AS DOUBLE) END as amount,
    CASE WHEN TRY_CAST(g.amount_total AS DOUBLE) > 0 THEN 'EUR' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    NULLIF(TRIM(g.activity), '') as funder_scheme,
    'lmt_spektras' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.start_year AS INT) as start_year,
    TRY_CAST(g.end_year AS INT) as end_year,
    g.lead as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN g.lead IS NOT NULL THEN array(g.lead) END as investigators,
    COALESCE(g.landing_page_url, 'https://spektras.lmt.lt/') as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'lmt_spektras' AND priority = 486;

-- Priority 486: direct-from-funder ingest of LMT's Spektras register.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    486 as priority
FROM openalex.awards.lmt_awards;

## Verification Queries

In [ ]:
%sql
-- Shape check (2.1.1): contract numbers, never application numbers (P-...).
SELECT COUNT(*) as total, COUNT(DISTINCT id) as distinct_ids,
       COUNT_IF(funder_award_id RLIKE '^S-[A-Z]+-[0-9]{2}-[0-9]+$') as s_form,
       COUNT_IF(funder_award_id RLIKE '^[A-Z]+-[0-9]+/[0-9]{4}$') as slash_form,
       COUNT_IF(funder_award_id RLIKE 'LMT-K') as esf_form,
       COUNT_IF(funder_award_id RLIKE '^P-') as application_form_should_be_0
FROM openalex.awards.lmt_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year, end_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.lmt_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.lmt_awards GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.lmt_awards WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- 6.4a frequency check.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.lmt_awards GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT
    COUNT(*) as total, COUNT(display_name) as has_title, COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt, ROUND(MAX(amount), 0) as max_amt, ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year, COUNT(lead_investigator) as has_pi,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.lmt_awards;

In [ ]:
%sql
-- Existing work-funder shells upgraded by this ingest (same funder + same contract number).
SELECT COUNT(*) as shell_rows_sharing_id, COUNT(DISTINCT s.id) as awards_with_shells
FROM openalex.awards.openalex_awards_raw s
JOIN openalex.awards.lmt_awards d ON d.id = s.id
WHERE s.priority = 0;

In [ ]:
%sql
-- 6.8: rows reached the shared raw table at the assigned priority.
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'lmt_spektras'
GROUP BY provenance, priority;